# ДЗ-15 (часть 1): LoRA fine-tuning чат-ассистента (Qwen2.5-1.5B)

Дообучаем **Qwen2.5-1.5B-Instruct** под качественные диалоги методом **QLoRA**
(LoRA поверх 4-битной модели) на сабсете датасета **lmsys/lmsys-chat-1m**.

## Зачем LoRA / PEFT
Полный fine-tuning LLM меняет все веса (миллиарды параметров) — это дорого по памяти и времени.
**PEFT** (Parameter-Efficient Fine-Tuning) обучает лишь малую добавку. **LoRA** вставляет в слои
внимания обучаемые низкоранговые матрицы `A·B` (ранг `r`), а исходные веса замораживает —
обучается ~0.1–1% параметров. **QLoRA** дополнительно квантует базовую модель в 4 бита, чтобы
влезть в один бесплатный GPU (Colab T4, 16 ГБ).

> ⚠️ **Нужен NVIDIA GPU.** `bitsandbytes` (4-бит) работает только на CUDA. Запускай этот ноутбук
> в **Google Colab** (Runtime → Change runtime type → T4 GPU) или на машине с CUDA.
> На CPU/Windows-Intel обучение не пойдёт.

## Шаг 1. Установка (в Colab)
Раскомментируй и выполни в Colab. Локально с CUDA ставь из `requirements.txt`.

In [ ]:
# !pip install -q -U torch transformers peft trl datasets accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), (
    "CUDA-GPU не найден. Запусти в Colab с T4 (Runtime -> Change runtime type -> GPU)."
)
print("GPU:", torch.cuda.get_device_name(0))

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

## Шаг 2. Загрузка модели в 4-битном виде (QLoRA)

In [ ]:
# 4-битная квантизация: модель занимает ~1.5 ГБ вместо ~3 ГБ, точность почти не страдает.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                 # nf4 — лучший формат для весов
    bnb_4bit_compute_dtype=torch.bfloat16,     # вычисления в bf16
    bnb_4bit_use_double_quant=True,            # двойная квантизация — ещё экономнее
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False                 # обязательно при обучении с gradient checkpointing
print("Модель загружена.")

## Шаг 3. Baseline ДО обучения
Сохраним ответ исходной модели на тестовый вопрос, чтобы потом сравнить с дообученной.

In [ ]:
def chat(model, question: str, max_new_tokens: int = 200) -> str:
    """Генерация ответа по chat-шаблону Qwen."""
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

TEST_Q = "Объясни простыми словами, чем отличается обучение с учителем от обучения без учителя."
baseline_answer = chat(model, TEST_Q)
print("=== ДО fine-tuning ===\n", baseline_answer)

## Шаг 4. Датасет: lmsys-chat-1m

`lmsys/lmsys-chat-1m` — **gated**: зайди на
[страницу датасета](https://huggingface.co/datasets/lmsys/lmsys-chat-1m), прими условия и
залогинься токеном (`HF_TOKEN`). Берём небольшой сабсет через streaming (не качаем весь 1M).

Если доступа к lmsys нет — функция автоматически переключится на открытый
`HuggingFaceH4/ultrachat_200k`.

In [ ]:
import os
from itertools import islice
from datasets import load_dataset, Dataset
from huggingface_hub import login

if os.getenv("HF_TOKEN"):
    login(os.environ["HF_TOKEN"])

N_SAMPLES = 2000        # сабсет для демонстрации; увеличь для лучшего качества
MAX_TURNS = 6           # ограничим длину диалогов

def to_messages_lmsys(row):
    # в lmsys поле 'conversation' = [{'role': 'user'/'assistant', 'content': ...}, ...]
    return [{"role": m["role"], "content": m["content"]} for m in row["conversation"][:MAX_TURNS]]

def to_messages_ultrachat(row):
    return [{"role": m["role"], "content": m["content"]} for m in row["messages"][:MAX_TURNS]]

def load_chat_subset():
    try:
        ds = load_dataset("lmsys/lmsys-chat-1m", split="train", streaming=True)
        rows = [to_messages_lmsys(r) for r in islice(ds, N_SAMPLES)]
        print(f"Загружено {len(rows)} диалогов из lmsys-chat-1m")
    except Exception as e:
        print(f"lmsys недоступен ({e}). Переключаюсь на ultrachat_200k.")
        ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True)
        rows = [to_messages_ultrachat(r) for r in islice(ds, N_SAMPLES)]
        print(f"Загружено {len(rows)} диалогов из ultrachat_200k")
    # оставляем только корректные диалоги (начинается с user, есть ответ ассистента)
    rows = [r for r in rows if len(r) >= 2 and r[0]["role"] == "user"]
    return rows

dialogues = load_chat_subset()
print("Пример диалога:", dialogues[0][:2])

## Шаг 5. Форматирование под chat-шаблон
SFTTrainer обучается на готовом тексте. Превращаем каждый диалог в строку через
`apply_chat_template` — так модель учится в том же формате, в котором её потом спрашивают.

In [ ]:
def format_example(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_texts = [{"text": format_example(d)} for d in dialogues]
train_ds = Dataset.from_list(train_texts)
print("Обучающих примеров:", len(train_ds))
print("\n--- Пример отформатированного текста ---\n", train_ds[0]["text"][:400])

## Шаг 6. Конфиг LoRA и обучение (SFTTrainer)

Параметры LoRA:
- `r` — ранг добавок (8/16/32): больше → выразительнее и больше обучаемых параметров;
- `lora_alpha` — масштаб добавок (обычно 2·r);
- `target_modules` — в какие слои вставлять LoRA (проекции attention + MLP);
- `lora_dropout` — регуляризация.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

model = prepare_model_for_kbit_training(model)   # включает gradient checkpointing для 4-бит

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

sft_config = SFTConfig(
    output_dir="qwen2.5-1.5b-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,    # эффективный батч = 8
    max_steps=60,                     # для демо; для реального качества — сотни/тысячи
    learning_rate=2e-4,
    logging_steps=10,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",         # экономный оптимизатор
    bf16=True,
    max_seq_length=1024,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
)
# доля обучаемых параметров — наглядно, что LoRA меняет ~доли процента
trainer.model.print_trainable_parameters()
trainer.train()

## Шаг 7. Сравнение ПОСЛЕ обучения
Тот же вопрос — но теперь отвечает модель с обученным LoRA-адаптером.

In [ ]:
ft_answer = chat(trainer.model, TEST_Q)
print("=== ДО fine-tuning ===\n", baseline_answer)
print("\n=== ПОСЛЕ fine-tuning ===\n", ft_answer)

## Шаг 8. Сохранение адаптера
Сохраняем только LoRA-адаптер (несколько МБ). В части 2 (`agent_demo.ipynb`) подгрузим его
поверх базовой модели.

In [ ]:
ADAPTER_DIR = "qwen2.5-1.5b-lora-adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Адаптер сохранён в", ADAPTER_DIR)

# (опционально) слить адаптер в полную модель для удобного инференса:
# from peft import PeftModel
# merged = trainer.model.merge_and_unload()
# merged.save_pretrained("qwen2.5-1.5b-merged")

## Выводы
- **LoRA/PEFT** позволил адаптировать модель, обучив доли процента параметров — это влезло
  в бесплатный GPU благодаря **QLoRA** (4-бит).
- Сравнение «до/после» на одном вопросе демонстрирует сдвиг стиля ответов к обучающим данным.
- Для реального качества: больше шагов (`max_steps`/эпохи), больше данных, валидация и подбор `r`.
- Дальше — `agent_demo.ipynb`: подключаем адаптер и даём модели **инструменты** (часть 2).